In [1]:
%load_ext autoreload
%autoreload 2

In [25]:
pd.options.display.float_format = '{:.3f}'.format

In [2]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append("/Users/harendrakumar/Documents/Loan_Prediction/")

In [3]:
from ML_Pipelines.ml.utils.data_loader import get_latest_partition_data
from ML_Pipelines.ml.pipelines.ingestion import IngestionPipeline
from ML_Pipelines.ml.pipelines import transformation
from ML_Pipelines.ml.pipelines.model_training import ModelTrainingPipeline
from ML_Pipelines.ml.utils import databalancer, data_loader

/Users/harendrakumar/.local/share/virtualenvs/harendrakumar-Fk8KTnIY/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from ML_Pipelines.ml.utils.utility import (
    drop_columns_having_nulls_above_threshold,
    fill_numeric_missing_values,
    fill_categorical_missing_values,
    fill_numeric_missing_values_using_interpolation,
    drop_columns_with_IDs,
    encode_cat_columns
)

In [41]:
from ML_Pipelines.ml.utils.feature_selectors import *

In [5]:
steps = [drop_columns_with_IDs,
         drop_columns_having_nulls_above_threshold, 
         fill_numeric_missing_values, 
         fill_categorical_missing_values,
        encode_cat_columns]

In [6]:
data_dir="../../data_folder/train"

In [7]:
df = transformation.TransformationPipeline(ingestion_pipeline=IngestionPipeline(data_dir=data_dir),
                                     transformation_steps=steps).run_transformation()

INFO:ML_Pipelines.ml.pipelines.ingestion:Latest partition file ../../data_folder/train/credit_train_20241027.csv loaded successfully.
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.pipelines.transformation:Applying transformation step: drop_columns_with_IDs
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.utils.utility:Dropping ID columns: ['Loan ID', 'Customer ID']
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.pipelines.transformation:Applying transformation step: drop_columns_having_nulls_above_threshold
INFO:ML_Pipelines.ml.pipelines.transformation:=================================
INFO:ML_Pipelines.ml.utils.utility:➡️ Dropping columns with more than 50.0% null values.
INFO:ML_Pipelines.ml.utils.utility:✅ Dropping columns with null percentage above 50.0: ['Months since last delinquent']
INFO:ML_Pipelines.ml.pipelines.transfo

In [26]:
df.head()

,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,1,445412.000,1,709.000,1167493.000,8,1,5,5214.740,17.200,6.000,1.000,228190.000,416746.000,1.000,0.000
1,1,262328.000,1,1076.456,1378276.560,1,1,3,33295.980,21.100,35.000,0.000,229976.000,850784.000,0.000,0.000
2,1,99999999.000,1,741.000,2231892.000,8,2,3,29200.530,14.900,18.000,1.000,297996.000,750090.000,0.000,0.000
3,1,347666.000,0,721.000,806949.000,3,2,3,8741.900,12.000,9.000,0.000,256329.000,386958.000,0.000,0.000
4,1,176220.000,1,1076.456,1378276.560,5,3,3,20639.700,6.100,15.000,0.000,253460.000,427174.000,0.000,0.000


In [9]:
model_train = ModelTrainingPipeline(data=df, target_column='Loan Status')

In [10]:
X_train, X_test, y_train, y_test  = model_train.divide_and_standardize_data()

INFO:ML_Pipelines.ml.pipelines.model_training:Dividing data with test size = 0.2


In [35]:
model = model_train.train_model(model_name='RandomForest', X_train=X_train, y_train=y_train)

INFO:ML_Pipelines.ml.pipelines.model_training:Starting model training pipeline.
INFO:ML_Pipelines.ml.pipelines.model_training:RandomForest model trained successfully.


In [36]:
model_train.evaluate_model(X_test=X_test, y_test=y_test)

INFO:ML_Pipelines.ml.pipelines.model_training:Confusion Matrix
INFO:ML_Pipelines.ml.pipelines.model_training:Classification Report 
:              precision    recall  f1-score   support

           0       1.00      0.21      0.34      4528
           1       0.81      1.00      0.90     15472

    accuracy                           0.82     20000
   macro avg       0.91      0.60      0.62     20000
weighted avg       0.85      0.82      0.77     20000

INFO:ML_Pipelines.ml.pipelines.model_training:ROC AUC Score: 0.7606
INFO:ML_Pipelines.ml.pipelines.model_training:Model evaluation metrics:
INFO:ML_Pipelines.ml.pipelines.model_training:Accuracy: 0.82065
INFO:ML_Pipelines.ml.pipelines.model_training:Precision: 0.8544045752662783
INFO:ML_Pipelines.ml.pipelines.model_training:Recall: 0.82065
INFO:ML_Pipelines.ml.pipelines.model_training:F1 Score: 0.7711493181020735


{'accuracy': 0.82065,
 'precision': 0.8544045752662783,
 'recall': 0.82065,
 'f1': 0.7711493181020735,
 'roc_auc_score': 0.7605543160607466,
 'confusion_matrix': array([[  941,  3587],
        [    0, 15472]])}

In [37]:
feature_names=  df.drop("Loan Status", axis=1).columns
feature_names

Index(['Current Loan Amount', 'Term', 'Credit Score', 'Annual Income',
       'Years in current job', 'Home Ownership', 'Purpose', 'Monthly Debt',
       'Years of Credit History', 'Number of Open Accounts',
       'Number of Credit Problems', 'Current Credit Balance',
       'Maximum Open Credit', 'Bankruptcies', 'Tax Liens'],
      dtype='object')

In [38]:
x_sample =df.drop('Loan Status', axis=1).iloc[:5000, :]
x_sample.shape

(5000, 15)

In [40]:
model_train.explain_prediction(X_sample=X_test, feature_names=feature_names)

array([[[-1.45724400e-01,  1.45724400e-01],
        [-1.34470388e-02,  1.34470388e-02],
        [-3.60163700e-02,  3.60163700e-02],
        ...,
        [-4.62601255e-03,  4.62601255e-03],
        [ 5.40339690e-05, -5.40339690e-05],
        [-1.42241645e-04,  1.42241645e-04]],

       [[ 2.55741350e-02, -2.55741350e-02],
        [-2.08426017e-02,  2.08426017e-02],
        [-4.31723068e-02,  4.31723068e-02],
        ...,
        [ 4.69233353e-03, -4.69233353e-03],
        [ 4.07711186e-05, -4.07711186e-05],
        [-8.21315194e-05,  8.21315194e-05]],

       [[ 1.82305716e-02, -1.82305716e-02],
        [-2.13218045e-02,  2.13218045e-02],
        [-3.31585181e-02,  3.31585181e-02],
        ...,
        [-9.63141016e-03,  9.63141016e-03],
        [ 5.20358014e-05, -5.20358014e-05],
        [-1.58260550e-04,  1.58260550e-04]],

       ...,

       [[ 2.44354596e-02, -2.44354596e-02],
        [-1.85544931e-02,  1.85544931e-02],
        [-4.03473812e-02,  4.03473812e-02],
        ...,
     

<Figure size 1000x800 with 0 Axes>